# 🤖 Final Production Modeling — Demand Forecasting

This notebook implements the winning configurations found during research:
1. **LightGBM**: Champion for granular item-level forecasting.
2. **SARIMA**: Champion for aggregate daily trends.

In [ ]:
%pip install lightgbm pmdarima statsmodels pandas numpy matplotlib seaborn scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, pickle, os
import gc
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
print("✅ Libraries imported.")

Note: you may need to restart the kernel to use updated packages.
✅ Libraries imported.


## 1. Load & Prepare Data

In [2]:
data_dir = 'preprocessed_data'
try:
    X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
    X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
    y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').iloc[:, 0]
    y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').iloc[:, 0]
    with open(f'{data_dir}/feature_names.pkl', 'rb') as f:
        FEATS = pickle.load(f)
    print("✅ Loaded data from parquet files.")
    print("\nFeatures loaded:")
    print(FEATS)
except Exception as e:
    print("⚠️ Parquet files not found. Please run preprocessing.ipynb first or ensure data is loaded.")
    raise e

✅ Loaded data from parquet files.

Features loaded:
['price_base', 'day', 'month', 'year', 'dayofweek', 'week', 'is_holiday', 'rolling_avg_quantity_w7', 'rolling_avg_quantity_w14', 'rolling_avg_quantity_w30', 'quarter', 'is_weekend', 'month_sin', 'month_cos', 'day_of_year', 'days_since_start', 'is_month_start', 'is_month_end', 'is_payday_near', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos', 'lag_7_quantity', 'lag_14_quantity', 'lag_28_quantity', 'lag_365_quantity', 'area', 'store_id', 'dept_name', 'class_name', 'subclass_name', 'item_type', 'format', 'division', 'city']


## 2. Cross-Validation (Time Series Split)

In [3]:
print("🚀 Running TimeSeriesSplit (3 Folds) on LightGBM...")
tscv = TimeSeriesSplit(n_splits=3)
fold = 1
for train_idx, val_idx in tscv.split(X_train):
    print(f"--- Fold {fold} ---")
    # To save memory, we're not deep-copying here since it's just for fast CV metric calculation
    X_cv_train = X_train.iloc[train_idx]
    y_cv_train = y_train.iloc[train_idx]
    X_cv_val = X_train.iloc[val_idx]
    y_cv_val = y_train.iloc[val_idx]
    
    cv_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, num_leaves=63, n_jobs=-1)
    cv_model.fit(X_cv_train, y_cv_train, eval_set=[(X_cv_val, y_cv_val)], callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)])
    
    cv_pred = cv_model.predict(X_cv_val)
    cv_rmse = np.sqrt(mean_squared_error(np.expm1(y_cv_val), np.expm1(np.maximum(cv_pred, 0))))
    print(f"Fold {fold} RMSE: {cv_rmse:.4f}")
    fold += 1
    del X_cv_train, y_cv_train, X_cv_val, y_cv_val, cv_model
    gc.collect()
print("✅ Cross-Validation Complete.")

🚀 Running TimeSeriesSplit (3 Folds) on LightGBM...
--- Fold 1 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043079 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3126
[LightGBM] [Info] Number of data points in the train set: 889415, number of used features: 36
[LightGBM] [Info] Start training from score 1.343605
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's l2: 0.20522
Fold 1 RMSE: 17.4535
--- Fold 2 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.110938 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3293
[LightGBM] [Info] Number of data points in the train set: 1778829, number of used features: 36

## 3. Train Item-Level Champion (LightGBM)

In [4]:
if 'X_train' in locals():
    print("🚀 Training Final Tuned LightGBM...")
    # Create a proper validation set from the LAST 15% of training data chronologically
    val_size = int(len(X_train) * 0.15)
    X_val = X_train.iloc[-val_size:].copy()
    y_val = y_train.iloc[-val_size:].copy()
    X_tr = X_train.iloc[:-val_size].copy()
    y_tr = y_train.iloc[:-val_size].copy()
    
    # Free memory
    del X_train, y_train
    gc.collect()

    lgb_model = lgb.LGBMRegressor(
        n_estimators=500, 
        learning_rate=0.05, 
        num_leaves=63,
        colsample_bytree=0.8, 
        subsample=0.8, 
        random_state=42, 
        verbose=-1,
        n_jobs=-1
    )

    # EARLY STOPPING FIX: Use X_val, y_val, not X_test
    lgb_model.fit(
        X_tr, y_tr, 
        eval_set=[(X_val, y_val)], 
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    lgb_pred = lgb_model.predict(X_test)
    
    # Aggregate back to daily quantities to compute MAPE properly (Apples to Apples)
    test_df = X_test.copy()
    test_df['pred_log'] = lgb_pred
    test_df['actual_log'] = y_test.values
    
    daily_lgb = test_df.groupby('date').agg(
        pred_qty=('pred_log', lambda x: np.expm1(np.maximum(x, 0)).sum()),
        actual_qty=('actual_log', lambda x: np.expm1(x).sum())
    ).reset_index()
    
    actuals = daily_lgb['actual_qty'].values
    preds = daily_lgb['pred_qty'].values
    mask = actuals > 0
    
    rmse_lgb = np.sqrt(mean_squared_error(actuals, preds))
    mae_lgb = mean_absolute_error(actuals, preds)
    mape_lgb = np.mean(np.abs((actuals[mask] - preds[mask]) / actuals[mask])) * 100
    r2_lgb = r2_score(actuals, preds)
    
    print(f"\n✅ LightGBM Aggregate Performance:")
    print(f"RMSE: {rmse_lgb:.4f}")
    print(f"MAE:  {mae_lgb:.4f}")
    print(f"MAPE: {mape_lgb:.4f}%")
    print(f"R²:   {r2_lgb:.4f}")


🚀 Training Final Tuned LightGBM...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.181258
[200]	valid_0's l2: 0.17776
[300]	valid_0's l2: 0.176846
[400]	valid_0's l2: 0.176125
[500]	valid_0's l2: 0.175565
Did not meet early stopping. Best iteration is:
[500]	valid_0's l2: 0.175565


KeyError: 'date'

## 4. Train Aggregate Champion (SARIMA)

In [ ]:
print("🚀 Training Aggregate SARIMA...")
try:
    train_raw = pd.read_parquet(f'{data_dir}/train_raw.parquet')
    test_raw = pd.read_parquet(f'{data_dir}/test_raw.parquet')
    
    sarima_train = train_raw.groupby('date')['quantity'].sum()
    sarima_test = test_raw.groupby('date')['quantity'].sum()
    
    # Free memory
    del train_raw, test_raw
    gc.collect()
    
    sarima_model = SARIMAX(
        sarima_train, 
        order=(1,1,2), 
        seasonal_order=(1,0,1,7), 
        enforce_stationarity=False, 
        enforce_invertibility=False
    ).fit(disp=False)
    
    sarima_pred = sarima_model.forecast(steps=len(sarima_test))
    sarima_pred = np.maximum(np.nan_to_num(sarima_pred, nan=0.0), 0)
    
    actuals_s = sarima_test.values
    preds_s = sarima_pred.values
    mask_s = actuals_s > 0
    
    rmse_s = np.sqrt(mean_squared_error(actuals_s, preds_s))
    mae_s = mean_absolute_error(actuals_s, preds_s)
    mape_s = np.mean(np.abs((actuals_s[mask_s] - preds_s[mask_s]) / actuals_s[mask_s])) * 100
    r2_s = r2_score(actuals_s, preds_s)
    
    print(f"\n✅ SARIMA Aggregate Performance:")
    print(f"RMSE: {rmse_s:.4f}")
    print(f"MAE:  {mae_s:.4f}")
    print(f"MAPE: {mape_s:.4f}%")
    print(f"R²:   {r2_s:.4f}")
    
    # Generate daily predictions to use in the plot
    daily_pred_s = pd.DataFrame({'date': sarima_test.index, 'pred_qty': preds_s})
except Exception as e:
    print("⚠️ Failed to train SARIMA:", e)
    daily_pred_s = None
    raise e

## 5. Final Performance Dashboard & Model Save

In [ ]:
if 'lgb_model' in locals():
    fig, ax = plt.subplots(1, 3, figsize=(22, 6))

    # Feature Importance
    imp = pd.Series(lgb_model.feature_importances_, index=X_tr.columns).sort_values()
    imp.tail(15).plot(kind='barh', ax=ax[0], color='steelblue')
    ax[0].set_title("LightGBM Feature Importance (Top 15)")
    ax[0].grid(axis='x', alpha=0.3)

    # Residual Distribution (Bias Check)
    residuals = actuals - preds
    print(f"\n🔍 BIAS CHECK: Mean Residual = {residuals.mean():.4f} (Positive = Under-forecasting, Negative = Over-forecasting)")
    
    sns.histplot(residuals, bins=50, ax=ax[1], kde=True, color='darkorange')
    ax[1].set_title("Error Distribution (Actuals - Preds)")
    # Fix x-axis to 1st and 99th percentiles
    p1, p99 = np.percentile(residuals, 1), np.percentile(residuals, 99)
    ax[1].set_xlim(p1, p99)
    ax[1].grid(alpha=0.3)

    # Forecast vs Actual Line Plot
    ax[2].plot(daily_lgb['date'], actuals, label='Actual Demand', color='black', alpha=0.7)
    ax[2].plot(daily_lgb['date'], preds, label='LightGBM Pred', color='steelblue', alpha=0.9)
    if daily_pred_s is not None:
        ax[2].plot(daily_pred_s['date'], daily_pred_s['pred_qty'], label='SARIMA Pred', color='darkorange', alpha=0.9, linestyle='--')
    ax[2].set_title("Forecast vs Actual (Daily Aggregates)")
    ax[2].legend()
    ax[2].grid(alpha=0.3)
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.savefig('final_model_performance.png', dpi=300)
    plt.show()

    # SAVE MODEL AFTER VALIDATION PASSES
    os.makedirs('trained_models', exist_ok=True)
    with open('trained_models/lgb_model_final.pkl', 'wb') as f:
        pickle.dump(lgb_model, f)
    if 'sarima_model' in locals():
        with open('trained_models/sarima_model_final.pkl', 'wb') as f:
            pickle.dump(sarima_model, f)
    print("\n✓ Models saved to trained_models/ AFTER validation passes.")